# 03 — Reproducing SMPL Eq.(1)

**Pedagogical notebook**: walk through SMPL's 5-line algebra.

PDF §6.6 calls out *"a simple implementation of a method described in the literature"*
as a working-model option.  This is that, with explicit references to Loper et al. 2015.

## SMPL Eq.(1)

$$M(\beta, \theta) = W\bigl(T_P(\beta, \theta),\; J(\beta),\; \theta,\; \mathcal{W}\bigr)$$

where

$$T_P(\beta, \theta) = \bar{T} + B_S(\beta) + B_P(\theta)$$

- $\bar{T}$: mean template (6890 vertices)
- $B_S(\beta)$: shape blend shapes — 10 PCA coefficients drive vertex offsets
- $B_P(\theta)$: pose blend shapes — 207-d encoding of (joint rotation - identity) drives offsets to fix LBS artifacts
- $J(\beta)$: joint regressor — 24×6890 sparse matrix that picks joints from the shaped mesh
- $\mathcal{W}$: linear blend skinning weights

## Inspect each term

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
import torch, numpy as np
from demo.src.smpl_forward import smpl_forward_eq1

verts, joints, model = smpl_forward_eq1(model_type='smplx')
print('Mean template T_bar:', model.v_template.shape)
print('Shape directions S :', model.shapedirs.shape, '   ← (V, 3, num_betas+num_expr)')
print('Joint regressor J  :', model.J_regressor.shape)
print('Blend weights W    :', model.lbs_weights.shape)
print('Posedirs P         :', model.posedirs.shape)

## Sanity check — shape blend visualization

In [ ]:
# Vary one beta coefficient ±2 σ and see body morph
import matplotlib.pyplot as plt
from demo.src.visualize import render_mesh_matplotlib

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
for ax, b0 in zip(axes, [-2.0, -1.0, 0.0, 1.0, 2.0]):
    betas = torch.zeros(1, 10); betas[0, 0] = b0
    out = model(betas=betas, body_pose=torch.zeros(1,63), global_orient=torch.zeros(1,3))
    img = render_mesh_matplotlib(out.vertices.squeeze().detach().numpy(), model.faces, azim=30, figsize=(4,4))
    ax.imshow(img); ax.set_title(f'β₀ = {b0:+.1f}'); ax.axis('off')
plt.suptitle('First shape PCA component (β₀): typically captures height')
plt.tight_layout()
plt.savefig('../results/shape_pca_beta0.png', dpi=120, bbox_inches='tight')
plt.show()

## Why this matters for HMR

HMR 2.0's neural network outputs `(β, θ, camera)` — only **85 numbers** —
that this same SMPL forward turns into a 3D mesh.  All the visual richness
of human pose lives inside the SMPL data structures we just inspected,
not inside the neural net.

That's the deep point of HMR (and why its 50ms inference is meaningful):
the network's job is just to produce a small, well-conditioned parameter
vector — the parametric body model does the rest.